# Preprocessing & Modeling
**Pipeline:** `ColumnTransformer` + `TransformedTargetRegressor` — preprocessing dan model jadi satu objek.

| Section | Isi |
|---|---|
| 1 | Setup & Load Data |
| 2 | Train-Test Split |
| 3 | Preprocessing Pipeline |
| 4 | Simpan Data |
| 5 | Model Baseline — Linear Regression |
| 6 | Polynomial Regression |
| 7 | Ridge Regression |
| 8 | Lasso Regression |
| 9 | Perbandingan Semua Model |

---
## 1. Setup & Load Data

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..'))  # agar bisa import src/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ── Import dari src/ ────────────────────────────────────────────────────────
from src.data.load_data        import load_raw, split_features_target, split_train_test
from src.features.build_features import (
    build_preprocessor, build_target_transformer,
    NUM_COLS, ORD_SILINDER, ORD_TAHUN, OHE_COLS, get_tahun_cats
)
from src.models.train          import (
    build_linear_pipeline, build_ridge_pipeline,
    build_lasso_pipeline, build_polynomial_pipeline,
    evaluate, save_model
)
from src.visualization.plots   import (
    save_fig, plot_actual_vs_pred, plot_residuals,
    plot_model_comparison, plot_train_test_gap, plot_feature_importance
)

pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
RANDOM_STATE = 42
TARGET = 'konsumsi_bbm'
print('Setup selesai.')

In [ ]:
df_raw = load_raw()
get_tahun_cats(df_raw)  # inisialisasi urutan tahun dari data

X, y = split_features_target(df_raw)
print(f'Data: {df_raw.shape} | Missing: {df_raw.isnull().sum().sum()}')
df_raw.head()

---
## 2. Train-Test Split
> Split **sebelum** preprocessing — parameter fit hanya dari training data.

In [ ]:
X_train, X_test, y_train, y_test = split_train_test(X, y)
print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')

---
## 3. Preprocessing Pipeline

```
Pipeline(
  preprocessor = ColumnTransformer([
      ('num')      → Median Imputer + Yeo-Johnson (standardize=True)
      ('ord_sil')  → OrdinalEncoder jumlah_silinder (3<4<5<6<8)
      ('ord_thn')  → OrdinalEncoder tahun_rilis (70..82)
      ('ohe')      → OneHotEncoder asal_pabrikan (drop=first)
  ]),
  model = TransformedTargetRegressor(
      regressor   = <model>,
      transformer = Yeo-Johnson target
  )
)
```

In [ ]:
# ── Outlier Handling (aktifkan untuk Phase 3) ──────────────────────────────
# Setelah Yeo-Johnson, clip nilai > ±3 std
#
# from sklearn.preprocessing import FunctionTransformer
#
# def clip_outliers(X, n_std=3):
#     X = X.copy()
#     for i in range(X.shape[1]):
#         mean, std = X[:, i].mean(), X[:, i].std()
#         X[:, i] = np.clip(X[:, i], mean - n_std*std, mean + n_std*std)
#     return X
#
# Tambahkan di num pipeline:
# ('clipper', FunctionTransformer(clip_outliers))
# ────────────────────────────────────────────────────────────────────────────

print('Outlier handling: nonaktif (Phase 1 baseline).')

---
## 4. Simpan Data Interim & Processed

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
import os

INTERIM_DIR   = os.path.join('..', 'data', 'interim')
PROCESSED_DIR = os.path.join('..', 'data', 'processed')
os.makedirs(INTERIM_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

TAHUN_CATS    = get_tahun_cats()
SILINDER_CATS = [['3','4','5','6','8']]

# ── Interim: encoded saja (tanpa Yeo-Johnson) ──────────────────────────────
_interim = ColumnTransformer([
    ('num',     SimpleImputer(strategy='median'), NUM_COLS),
    ('ord_sil', OrdinalEncoder(categories=SILINDER_CATS, handle_unknown='use_encoded_value', unknown_value=-1), ORD_SILINDER),
    ('ord_thn', OrdinalEncoder(categories=TAHUN_CATS,    handle_unknown='use_encoded_value', unknown_value=-1), ORD_TAHUN),
    ('ohe',     OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), OHE_COLS),
], remainder='drop')

X_interim = _interim.fit_transform(X)
ohe_names = _interim.named_transformers_['ohe'].get_feature_names_out(OHE_COLS)
interim_cols = NUM_COLS + ORD_SILINDER + ORD_TAHUN + list(ohe_names)
df_interim = pd.DataFrame(X_interim, columns=interim_cols)
df_interim[TARGET] = y.values
df_interim.to_csv(os.path.join(INTERIM_DIR, 'data_encoded.csv'), index=False)

# ── Processed: full Yeo-Johnson ────────────────────────────────────────────
_prep = build_preprocessor()
_prep.fit(X_train)
ohe_names_proc = _prep.named_transformers_['ohe'].get_feature_names_out(OHE_COLS)
feature_names  = NUM_COLS + ORD_SILINDER + ORD_TAHUN + list(ohe_names_proc)

X_all_prep = _prep.transform(X)
df_processed = pd.DataFrame(X_all_prep, columns=feature_names)
_tgt = build_target_transformer()
_tgt.fit(y_train.values.reshape(-1, 1))
df_processed[TARGET + '_transformed'] = _tgt.transform(y.values.reshape(-1, 1)).ravel()
df_processed[TARGET + '_original']    = y.values
df_processed.to_csv(os.path.join(PROCESSED_DIR, 'data_preprocessed.csv'), index=False)

print(f'Interim  → {INTERIM_DIR}/data_encoded.csv    {df_interim.shape}')
print(f'Processed→ {PROCESSED_DIR}/data_preprocessed.csv {df_processed.shape}')
print(f'Fitur ({len(feature_names)}): {feature_names}')

---
## 5–8. Modeling

| Model | Penalti | Kelebihan |
|---|---|---|
| Linear Regression | Tidak ada | Baseline, sederhana |
| Polynomial (degree=2) | Tidak ada | Tangkap non-linearitas |
| Ridge (L2) | λΣwᵢ² | Atasi multikolinearitas |
| Lasso (L1) | λΣ\|wᵢ\| | Seleksi fitur otomatis |

**Metrik:** RMSE & MAE (satuan mpg, sudah inverse-transform) | R² | CV RMSE (5-fold)

In [ ]:
all_results  = []
all_pipelines = {}

def run(name, pipeline):
    result = evaluate(name, pipeline, X_train, y_train, X_test, y_test)
    y_pred = result.pop('_y_pred')
    pipe   = result.pop('_pipeline')
    print(f"  RMSE Train={result['RMSE Train']:.4f} | RMSE Test={result['RMSE Test']:.4f} "
          f"| R²={result['R² Test']:.4f} | CV={result['CV RMSE']:.4f}")
    gap = result['RMSE Test'] - result['RMSE Train']
    print(f"  Gap train/test: {gap:+.4f} "
          f"{'⚠️ overfitting' if gap > 1.5 else '✅ wajar'}")
    all_results.append(result)
    all_pipelines[name] = pipe
    return pipe, y_pred, result

In [ ]:
print('── Linear Regression (Baseline) ──')
lr_pipe, lr_pred, lr_res = run('Linear Regression', build_linear_pipeline())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
plot_actual_vs_pred(y_test.values, lr_pred, 'Linear Regression', ax=axes[0], show=False)
plot_residuals(y_test.values, lr_pred, 'Linear Regression', ax=axes[1], show=False)
plt.tight_layout()
save_fig('model_linear_regression')
plt.show()

lr_coef = pd.Series(lr_pipe.named_steps['model'].regressor_.coef_, index=feature_names).sort_values(key=abs, ascending=False)
plot_feature_importance(lr_coef, 'Koefisien Linear Regression', figname='model_linear_koefisien')

In [ ]:
print('── Polynomial Regression (degree=2) ──')
poly_pipe, poly_pred, poly_res = run('Polynomial Regression (degree=2)', build_polynomial_pipeline())
n_before = build_preprocessor().fit_transform(X_train).shape[1]
n_after  = poly_pipe.named_steps['poly'].n_output_features_
print(f'Fitur: {n_before} → {n_after} (setelah poly expansion)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
plot_actual_vs_pred(y_test.values, poly_pred, 'Polynomial (deg=2)', color_idx=2, ax=axes[0], show=False)
plot_residuals(y_test.values, poly_pred, 'Polynomial (deg=2)', color_idx=3, ax=axes[1], show=False)
plt.tight_layout()
save_fig('model_polynomial_regression')
plt.show()

In [ ]:
print('── Ridge Regression ──')
ridge_pipe, ridge_pred, ridge_res = run('Ridge Regression', build_ridge_pipeline())
print(f'Alpha terbaik Ridge: {ridge_pipe.named_steps["model"].regressor_.alpha_:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
plot_actual_vs_pred(y_test.values, ridge_pred, 'Ridge Regression', color_idx=4, ax=axes[0], show=False)
plot_residuals(y_test.values, ridge_pred, 'Ridge Regression', color_idx=5, ax=axes[1], show=False)
plt.tight_layout()
save_fig('model_ridge_regression')
plt.show()

# Perbandingan koefisien Linear vs Ridge
coef_compare = pd.DataFrame({
    'Linear': lr_pipe.named_steps['model'].regressor_.coef_,
    'Ridge' : ridge_pipe.named_steps['model'].regressor_.coef_
}, index=feature_names).sort_values('Ridge', key=abs, ascending=False)
print('\nPerbandingan koefisien Linear vs Ridge:')
display(coef_compare)

In [ ]:
print('── Lasso Regression ──')
lasso_pipe, lasso_pred, lasso_res = run('Lasso Regression', build_lasso_pipeline())
lasso_reg = lasso_pipe.named_steps['model'].regressor_
print(f'Alpha terbaik Lasso: {lasso_reg.alpha_:.6f}')
print(f'Fitur di-nol-kan   : {(lasso_reg.coef_ == 0).sum()} dari {len(feature_names)}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
plot_actual_vs_pred(y_test.values, lasso_pred, 'Lasso Regression', color_idx=0, ax=axes[0], show=False)
plot_residuals(y_test.values, lasso_pred, 'Lasso Regression', color_idx=1, ax=axes[1], show=False)
lasso_coef = pd.Series(lasso_reg.coef_, index=feature_names).sort_values()
plot_feature_importance(lasso_coef, 'Koefisien Lasso (0=di-drop)', ax=axes[2] if False else None,
                        figname=None)

active  = lasso_coef[lasso_coef != 0]
dropped = lasso_coef[lasso_coef == 0]
print(f'Dipertahankan ({len(active)}): {active.index.tolist()}')
print(f'Di-drop       ({len(dropped)}): {dropped.index.tolist()}')

---
## 9. Perbandingan Semua Model

In [ ]:
results_df = pd.DataFrame(all_results).set_index('Model')
display(results_df)
print(f"\n🏆 RMSE Test terbaik : {results_df['RMSE Test'].idxmin()} ({results_df['RMSE Test'].min():.4f} mpg)")
print(f"🏆 R² Test terbaik   : {results_df['R² Test'].idxmax()} ({results_df['R² Test'].max():.4f})")
print(f"🏆 CV RMSE terbaik   : {results_df['CV RMSE'].idxmin()} ({results_df['CV RMSE'].min():.4f} mpg)")

In [ ]:
plot_model_comparison(results_df, figname='model_comparison_bar')
plot_train_test_gap(results_df, figname='model_train_test_gap')